In [ ]:
!pip -q install transformers torch accelerate

In [ ]:
import os
import zipfile
import struct
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, GPT2LMHeadModel

# -----------------------------
# Configuration
# -----------------------------

DRAFT_MODEL = "distilgpt2"
TARGET_MODEL = "gpt2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(DRAFT_MODEL)
tokenizer.pad_token = tokenizer.eos_token

draft_model = GPT2LMHeadModel.from_pretrained(
    DRAFT_MODEL
).to(device).eval()

target_model = GPT2LMHeadModel.from_pretrained(
    TARGET_MODEL
).to(device).eval()

PROMPT = "Artificial intelligence is"

prompt_ids = tokenizer.encode(
    PROMPT,
    return_tensors="pt"
)[0].to(device)

K = 8
ROUNDS = 5

# datasets to generate
DATASETS = [
    500000,
    600000
]

CHUNK = 1024


# -------------------------------------------------------
# Draft generation
# -------------------------------------------------------

def draft_generate(context_ids, B):

    all_candidates = []
    all_logq = []

    for start in range(0, B, CHUNK):

        b = min(CHUNK, B-start)

        cur = context_ids.unsqueeze(0).repeat(b,1)

        cands=[]
        lqs=[]

        past=None

        with torch.inference_mode():

            for _ in range(K):

                out = draft_model(
                    cur,
                    past_key_values=past,
                    use_cache=True
                )

                logits = out.logits[:,-1,:]
                past = out.past_key_values

                lp = F.log_softmax(logits,dim=-1)

                tok = torch.multinomial(
                    torch.exp(lp),
                    1
                )

                cands.append(tok)
                lqs.append(lp.gather(1,tok))

                cur = tok

        all_candidates.append(torch.cat(cands,1))
        all_logq.append(torch.cat(lqs,1))

    return (
        torch.cat(all_candidates,0),
        torch.cat(all_logq,0)
    )


# -------------------------------------------------------
# Target scoring
# -------------------------------------------------------

def target_score(context_ids,candidates):

    B = candidates.shape[0]
    ctx = context_ids.shape[0]

    outputs=[]

    for start in range(0,B,CHUNK):

        end=min(start+CHUNK,B)

        chunk=candidates[start:end]

        full=torch.cat([
            context_ids.unsqueeze(0).repeat(end-start,1),
            chunk
        ],1)

        with torch.no_grad():
            logits=target_model(full).logits

        pred=logits[:,ctx-1:ctx+K-1,:]

        lp=F.log_softmax(pred,dim=-1)

        outputs.append(
            lp.gather(
                2,
                chunk.unsqueeze(-1)
            ).squeeze(-1)
        )

    return torch.cat(outputs,0)


# -------------------------------------------------------
# Write binary
# -------------------------------------------------------

def write_file(filename,B):

    print("="*60)
    print("Generating",filename)

    with open(filename,"wb") as f:

        f.write(
            struct.pack(
                "<iiii",
                ROUNDS,
                B,
                K,
                len(prompt_ids)
            )
        )

        f.write(
            struct.pack(
                f"<{len(prompt_ids)}i",
                *prompt_ids.cpu().tolist()
            )
        )

        for r in range(ROUNDS):

            print(f"Round {r+1}/{ROUNDS}")

            cand,logq = draft_generate(prompt_ids,B)

            logp = target_score(prompt_ids,cand)

            ratio = torch.exp(
                logp-logq
            ).clamp(max=1).mean()

            print("Mean p/q =",float(ratio))

            f.write(struct.pack("<i",r))

            cand_cpu=cand.cpu()
            logq_cpu=logq.cpu()
            logp_cpu=logp.cpu()

            for b in range(B):

                for k in range(K):

                    f.write(
                        struct.pack(
                            "<iff",
                            int(cand_cpu[b,k]),
                            float(logq_cpu[b,k]),
                            float(logp_cpu[b,k])
                        )
                    )

    print(filename,"done.")

Using: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (4 outputs left out)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
for B in DATASETS:
    write_file(f"rounds_{B}.bin",B)

Generating rounds_500000.bin
Round 1/5
Mean p/q = 0.77922523021698
Round 2/5
Mean p/q = 0.7792634963989258
Round 3/5


In [ ]:
zip_name="benchmark_datasets.zip"

with zipfile.ZipFile(zip_name,"w",zipfile.ZIP_DEFLATED) as z:

    for B in DATASETS:

        z.write(
            f"rounds_{B}.bin"
        )

print("Created",zip_name)

Created benchmark_datasets.zip


In [ ]:
from google.colab import files

files.download("benchmark_datasets.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>